# Hallucination Rates of GPT-4o (LLM Self-Evaluation)

Computes the hallucination-rate metrics defined in the paper from the self-evaluation counts
produced by `generate_and_self_evaluate.py`, and analyses them by citation bin and field.

For every paper the self-evaluation returns `X, Y, Z`:
**X = M** (matched authors), **Y = H** (hallucinated authors), **Z = U** (unretrieved true authors).

| Metric | Definition |
|---|---|
| HR1, max-normalized error rate | 1 - M / max(M + H, M + U) |
| HR2, Jaccard hallucination rate (primary) | (H + U) / (M + H + U) |
| HR3, harmonic hallucination rate | 2HU / ((M + H)U + (M + U)H) |
| HR4, hallucination-only rate | H / (M + H) |
| HR5, omission-only rate | U / (M + U) |

Contents: (1) metrics per paper, (2) statistics per citation bin, (3) per field x citation bin with 95%
confidence intervals, (4) per field, (5) question length per stratum, (6) regression and mixed-effects models.

**Input:** `results/gpt4o_recall_llm_selfeval/self_evaluated_answers.csv`, `data/eval_set/eval_set_9108.csv`
**Output:** tables and figures in `results/gpt4o_recall_llm_selfeval/`.

In [ ]:
import ast
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sys.path.insert(0, str(Path.cwd().parent))  # run this notebook from its own folder
import config

OUT_DIR = config.GPT4O_RECALL_DIR
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
BIN_ORDER = config.CITATION_BIN_LABELS
FIELD_ORDER = config.FIELDS

## 1. Per-paper metrics

In [ ]:
df = pd.read_csv(config.GPT4O_SELF_EVAL_CSV)
n_total = len(df)
df = df[df["evaluation"] != "None, None, None"].copy()
print(f"{len(df)} of {n_total} papers have a valid self-evaluation")

df[["M", "H", "U"]] = df["evaluation"].str.split(", ", expand=True).astype(int)


def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Element-wise ratio that returns 0 where the denominator is 0."""
    return (numerator / denominator.replace(0, np.nan)).fillna(0.0)


M, H, U = df["M"], df["H"], df["U"]
df["hr1"] = 1 - safe_ratio(M, np.maximum(M + H, M + U))
df["hr2"] = safe_ratio(H + U, M + H + U)
df["hr3"] = safe_ratio(2 * H * U, (M + H) * U + (M + U) * H)
df["hr4"] = safe_ratio(H, M + H)
df["hr5"] = safe_ratio(U, M + U)
df["perfect_recall"] = ((H == 0) & (U == 0)).astype(int)

df["citation_bin"] = pd.Categorical(df["citation_bin"], categories=BIN_ORDER, ordered=True)
df["field"] = pd.Categorical(df["field"], categories=FIELD_ORDER, ordered=True)
df.to_csv(OUT_DIR / "hallucination_rates_per_paper.csv", index=False)
df[["hr1", "hr2", "hr3", "hr4", "hr5"]].describe()

Correlation between the metrics (the paper reports that they are highly correlated).

In [ ]:
df[["hr1", "hr2", "hr3", "hr4", "hr5"]].corr(method="pearson").round(3)

## 2. Statistics per citation bin

In [ ]:
HR_COLS = ["hr1", "hr2", "hr3", "hr4", "hr5"]


def describe_groups(data: pd.DataFrame, by: list[str], metrics: list[str]) -> pd.DataFrame:
    """Count, mean, std, quantiles and a normal-approximation 95% CI of the mean for each group."""
    parts = []
    for metric in metrics:
        grouped = data.groupby(by, observed=True)[metric]
        table = grouped.agg(["count", "mean", "std", "min", "median", "max"])
        table["q25"] = grouped.quantile(0.25)
        table["q75"] = grouped.quantile(0.75)
        se = table["std"] / np.sqrt(table["count"])
        table["ci95_lower"] = table["mean"] - 1.96 * se
        table["ci95_upper"] = table["mean"] + 1.96 * se
        table.columns = [f"{col}_{metric}" for col in table.columns]
        parts.append(table)
    return pd.concat(parts, axis=1).reset_index()


by_bin = describe_groups(df, ["citation_bin"], HR_COLS)
by_bin.to_csv(OUT_DIR / "hr_by_citation_bin.csv", index=False)
by_bin[["citation_bin", "count_hr2"] + [f"mean_{m}" for m in HR_COLS]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for metric, marker in zip(HR_COLS[:3], ["o", "s", "^"]):
    ax.plot(by_bin["citation_bin"].astype(str), by_bin[f"mean_{metric}"], marker=marker, label=metric.upper())
ax.set_title("Mean HR across Citation Bins (GPT-4o, self-evaluation)", fontsize=14)
ax.set_xlabel("Citation Bin")
ax.set_ylabel("Mean HR")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "hr_by_citation_bin.png", dpi=300, bbox_inches="tight")
plt.show()

Spearman correlation between citation count and HR2 at the paper level.

In [ ]:
rho, p = stats.spearmanr(df["citation"], df["hr2"])
print(f"Spearman rho(citation, HR2) = {rho:.3f}, p = {p:.3g}")

## 3. Per field x citation bin, with 95% confidence intervals

The confidence intervals are used to check that ~100 papers per stratum give stable estimates.

In [ ]:
by_field_bin = describe_groups(df, ["field", "citation_bin"], HR_COLS[:3])
by_field_bin.to_csv(OUT_DIR / "hr_by_field_and_bin.csv", index=False)

for metric in HR_COLS[:3]:
    width = by_field_bin[f"ci95_upper_{metric}"] - by_field_bin[f"ci95_lower_{metric}"]
    print(f"{metric.upper()}: {len(by_field_bin)} strata | mean std = {by_field_bin[f'std_{metric}'].mean():.3f} | "
          f"mean CI width = {width.mean():.3f} | share of CI widths > 0.1: {(width > 0.1).mean():.2f}, "
          f"> 0.2: {(width > 0.2).mean():.2f}")

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(FIELD_ORDER) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), sharey=True)
for ax, field in zip(axes.flatten(), FIELD_ORDER):
    subset = by_field_bin[by_field_bin["field"] == field]
    for metric, marker in zip(HR_COLS[:3], ["o", "s", "^"]):
        ax.plot(subset["citation_bin"].astype(str), subset[f"mean_{metric}"], marker=marker, label=metric.upper())
    ax.set_title(field.title(), fontsize=12)
    ax.set_xlabel("Citation Bin")
    ax.set_ylabel("Mean HR")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle("Mean HR across Citation Bins by Field", fontsize=16, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "hr_by_field_and_bin.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Per field

In [ ]:
by_field = describe_groups(df, ["field"], HR_COLS[:3])
by_field.to_csv(OUT_DIR / "hr_by_field.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
for metric, marker in zip(HR_COLS[:3], ["o", "s", "^"]):
    ax.plot(by_field["field"].astype(str), by_field[f"mean_{metric}"], marker=marker, label=metric.upper())
ax.set_title("Mean HR across Fields", fontsize=14)
ax.set_xlabel("Field")
ax.set_ylabel("Mean HR")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "hr_by_field.png", dpi=300, bbox_inches="tight")
plt.show()
by_field[["field"] + [f"mean_{m}" for m in HR_COLS[:3]]].round(3)

## 5. Question length per stratum

In [ ]:
df["question_length"] = df["question"].str.len()
length_table = (
    df.groupby(["field", "citation_bin"], observed=True)["question_length"]
    .agg(count="count", mean_length="mean", std_length="std", min_length="min", max_length="max")
    .reset_index()
)
length_table.to_csv(OUT_DIR / "question_length_by_field_and_bin.csv", index=False)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
for ax, field in zip(axes.flatten(), FIELD_ORDER):
    subset = length_table[length_table["field"] == field]
    labels = subset["citation_bin"].astype(str)
    ax.bar(labels, subset["mean_length"], color="#4C97F7", alpha=0.8)
    ax.errorbar(labels, subset["mean_length"], yerr=subset["std_length"], fmt="none", ecolor="gray", capsize=3)
    ax.set_title(field.title(), fontsize=12)
    ax.set_xlabel("Citation Bin")
    ax.set_ylabel("Mean Question Length")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)
fig.suptitle("Mean Question Length across Citation Bins by Field", fontsize=16, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "question_length_by_field_and_bin.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Multivariate regression and mixed-effects models

Covariates: log2(citation + 1), author count, title length, publication year, and field.
Models are fitted for HR2 and HR4:

* OLS with field dummies, plus the R² gained by adding citation count;
* linear mixed-effects models with a random intercept per field, and with an additional random slope
  for log citation (compared with a likelihood-ratio test);
* a fractional-logit GLM (binomial family, logit link).

In [ ]:
papers = pd.read_csv(config.EVAL_SET_CSV, usecols=["paperId", "title", "year", "full_name_list"])
data = df.merge(papers, on="paperId", how="inner")
data["field"] = data["field"].astype(str)
data["title_length"] = data["title"].str.len()
data["author_count"] = data["full_name_list"].apply(lambda s: len(ast.literal_eval(s)))
data["log_citation"] = np.log2(data["citation"] + 1)
print(f"{len(data)} papers in the regression data")

field_dummies = pd.get_dummies(data["field"], drop_first=True).astype(float)
X = sm.add_constant(pd.concat([data[["log_citation", "author_count", "title_length", "year"]], field_dummies], axis=1))
X_no_citation = X.drop(columns=["log_citation"])

In [ ]:
for target in ["hr2", "hr4"]:
    full_model = sm.OLS(data[target], X).fit()
    reduced_model = sm.OLS(data[target], X_no_citation).fit()
    print(f"==================== OLS: {target.upper()} ====================")
    print(full_model.summary())
    print(f"Full model R² = {full_model.rsquared:.4f} | without citation R² = {reduced_model.rsquared:.4f} | "
          f"additional R² from citation = {full_model.rsquared - reduced_model.rsquared:.4f}\n")

In [ ]:
formula_rhs = "log_citation + author_count + title_length + year"
for target in ["hr2", "hr4"]:
    random_intercept = smf.mixedlm(f"{target} ~ {formula_rhs}", data=data, groups=data["field"]).fit(reml=False)
    random_slope = smf.mixedlm(f"{target} ~ {formula_rhs}", data=data, groups=data["field"],
                               re_formula="~log_citation").fit(reml=False)
    print(f"==================== Mixed effects: {target.upper()} ====================")
    print(random_intercept.summary())
    print(random_slope.summary())

    # Likelihood-ratio test: the random-slope model adds 2 parameters (slope variance and covariance).
    lr_stat = 2 * (random_slope.llf - random_intercept.llf)
    p_value = stats.chi2.sf(lr_stat, 2)
    print(f"AIC random intercept = {random_intercept.aic:.2f} | AIC random slope = {random_slope.aic:.2f}")
    print(f"LR statistic = {lr_stat:.2f}, p = {p_value:.6f}\n")

In [ ]:
for target in ["hr2", "hr4"]:
    # Clip to (0, 1): HR values of exactly 0 or 1 would give an infinite logit.
    y = data[target].clip(0.001, 0.999)
    glm = sm.GLM(y, X, family=sm.families.Binomial(link=sm.families.links.Logit())).fit()
    print(f"==================== Fractional logit GLM: {target.upper()} ====================")
    print(glm.summary())
    odds_ratio = np.exp(glm.params["log_citation"])
    print(f"Odds ratio per doubling of citations = {odds_ratio:.3f} ({odds_ratio - 1:+.1%} change in odds)\n")